# Fine-tune Llama 3.2 for Urdu on Kaggle

A complete, production-ready pipeline for fine-tuning Meta's Llama 3.2 on Urdu instruction data using QLoRA.

**Notebook Features:**
- Real Urdu translation data from Hugging Face
- Memory-efficient QLoRA fine-tuning
- Validation metrics and evaluation
- Inference examples
- Model export for deployment

**Expected runtime:** ~30-45 minutes on Kaggle T4 GPU

**Author:** danielosinor123-gif

## ⚙️ Setup & Configuration

In [ ]:
# Configuration
CONFIG = {
    "model_name": "meta-llama/Llama-3.2-1B-Instruct",  # 1B for faster Kaggle training
    "dataset_size": 1500,  # Number of examples to use
    "train_split": 0.9,
    "batch_size": 2,
    "gradient_accumulation": 4,
    "learning_rate": 2e-4,
    "num_epochs": 1,
    "max_seq_length": 2048,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
}

print("📋 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 📦 Install Dependencies

In [ ]:
import subprocess
import sys

packages = [
    "accelerate==0.27.0",
    "bitsandbytes==0.42.0",
    "datasets==2.20.0",
    "peft==0.11.0",
    "transformers==4.41.0",
    "trl==0.8.6",
    "evaluate==0.4.1",
    "rouge-score==0.1.2",
    "peft",
]

print("Installing dependencies...\n")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("\n✅ All dependencies installed!")

## 📥 Download & Prepare Urdu Data

In [ ]:
from datasets import load_dataset
import json
import random
from pathlib import Path

# Create data directory
Path('data').mkdir(exist_ok=True)

print("Downloading Urdu-English translation dataset from Hugging Face...")
print("Dataset: Helsinki-NLP/opus_books (English-Urdu)\n")

dataset = load_dataset("Helsinki-NLP/opus_books", "en-ur", split="train")

print(f"✓ Dataset loaded: {len(dataset):,} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n📖 First example:")
print(f"  English: {dataset[0]['translation']['en'][:80]}...")
print(f"  Urdu: {dataset[0]['translation']['ur'][:80]}...")

## 🔄 Convert to Instruction Format

In [ ]:
def create_urdu_instruction(example):
    """
    Convert EN->UR translation pairs into diverse instruction formats.
    This creates varied prompts to improve model generalization.
    """
    en_text = example['translation']['en']
    ur_text = example['translation']['ur']
    
    # Diverse instruction templates
    templates = [
        f"اردو میں ترجمہ کریں:\n{en_text}",
        f"Translate to Urdu:\n{en_text}",
        f"اس انگریزی جملے کو اردو میں بدلیں:\n{en_text}",
        f"انگریزی سے اردو: {en_text}",
    ]
    
    prompt = random.choice(templates)
    
    return {
        'prompt': prompt,
        'response': ur_text
    }

print(f"Converting {CONFIG['dataset_size']} examples to instruction format...\n")

instructions = []
for i, example in enumerate(dataset):
    if i >= CONFIG['dataset_size']:
        break
    instr = create_urdu_instruction(example)
    instructions.append(instr)

print(f"✓ Created {len(instructions):,} instruction examples\n")

# Show examples
print("📝 Sample training examples:\n")
for i in range(3):
    print(f"Example {i+1}:")
    print(f"  Prompt: {instructions[i]['prompt'][:70]}...")
    print(f"  Response: {instructions[i]['response'][:70]}...\n")

## 💾 Save & Split Data

In [ ]:
from sklearn.model_selection import train_test_split
import os

# Split into train/val
train_data, val_data = train_test_split(
    instructions,
    test_size=1-CONFIG['train_split'],
    random_state=42
)

# Save training data
train_path = 'data/urdu_train.jsonl'
with open(train_path, 'w', encoding='utf-8') as f:
    for item in train_data:
        json.dump(item, f, ensure_ascii=False)
        f.write('\n')

# Save validation data
val_path = 'data/urdu_val.jsonl'
with open(val_path, 'w', encoding='utf-8') as f:
    for item in val_data:
        json.dump(item, f, ensure_ascii=False)
        f.write('\n')

print(f"✓ Train split: {len(train_data):,} examples → {train_path}")
print(f"  File size: {os.path.getsize(train_path) / (1024**2):.2f} MB")
print(f"\n✓ Validation split: {len(val_data):,} examples → {val_path}")
print(f"  File size: {os.path.getsize(val_path) / (1024**2):.2f} MB")

## 🤖 Load Model with 4-bit Quantization

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_name = CONFIG['model_name']
print(f"Loading {model_name}...\n")

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

print(f"✓ Model loaded successfully")
print(f"  Model dtype: {model.dtype}")
print(f"  Model device: {next(model.parameters()).device}")
print(f"  Tokenizer vocab size: {len(tokenizer)}")

## 🎯 Configure QLoRA

In [ ]:
# Prepare model for 4-bit training
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=CONFIG['lora_dropout'],
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

print("✓ QLoRA configured\n")
model.print_trainable_parameters()

## 📊 Prepare Datasets

In [ ]:
from datasets import load_dataset

# Load datasets
train_dataset = load_dataset("json", data_files=train_path, split="train")
val_dataset = load_dataset("json", data_files=val_path, split="train")

def format_chat_template(example):
    """
    Format prompt-response pairs using Llama 3.2 chat template.
    """
    prompt = example["prompt"]
    response = example["response"]
    
    text = (
        "<|begin_of_text|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{prompt}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{response}<|eot_id|>"
    )
    return {"text": text}

# Apply formatting
train_dataset = train_dataset.map(
    format_chat_template,
    remove_columns=["prompt", "response"],
    desc="Formatting training data"
)
val_dataset = val_dataset.map(
    format_chat_template,
    remove_columns=["prompt", "response"],
    desc="Formatting validation data"
)

print(f"✓ Training dataset: {len(train_dataset):,} examples")
print(f"✓ Validation dataset: {len(val_dataset):,} examples\n")

# Show example
print("📋 Formatted example (first 300 chars):")
print(train_dataset[0]["text"][:300])

## ⚡ Configure Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./urdu-llama-3.2-qLoRA",
    per_device_train_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation'],
    learning_rate=CONFIG['learning_rate'],
    num_train_epochs=CONFIG['num_epochs'],
    logging_steps=10,
    save_steps=100,
    eval_steps=50,
    save_total_limit=2,
    evaluation_strategy="steps",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    report_to="none",
    max_grad_norm=0.3,
    seed=42,
)

print("✓ Training configuration set\n")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size (per device): {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Epochs: {training_args.num_train_epochs}")

## 🚀 Train the Model

In [ ]:
from trl import SFTTrainer

print("Initializing trainer...\n")

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    max_seq_length=CONFIG['max_seq_length'],
    args=training_args,
    packing=False,
)

print("🔥 Starting training...\n")
trainer.train()

print("\n✅ Training complete!")

## 💾 Save Fine-tuned Model

In [ ]:
output_dir = "urdu-llama-3.2-qLoRA-final"

print(f"Saving model to {output_dir}...\n")

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✓ LoRA adapter saved")
print(f"✓ Tokenizer saved")
print(f"✓ Model ready for inference")

## 🧪 Test Inference

In [ ]:
from peft import AutoPeftModelForCausalLM

# Load fine-tuned model
print("Loading fine-tuned model for inference...\n")

model = AutoPeftModelForCausalLM.from_pretrained(
    output_dir,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(output_dir)

def generate_urdu_text(prompt: str, max_tokens: int = 150) -> str:
    """
    Generate Urdu text from a prompt using the fine-tuned model.
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

# Test prompts
test_prompts = [
    "اردو میں ترجمہ کریں:\nHello, how are you?",
    "Translate to Urdu:\nThe weather is beautiful today.",
    "اس انگریزی جملے کو اردو میں بدلیں:\nGood morning, have a great day!",
]

print("="*70)
print("🎯 INFERENCE RESULTS")
print("="*70 + "\n")

for i, prompt in enumerate(test_prompts, 1):
    response = generate_urdu_text(prompt)
    print(f"Example {i}:")
    print(f"  Question: {prompt}")
    print(f"  Response: {response}")
    print()

## 📈 Training Metrics Summary

In [ ]:
import json
from pathlib import Path

# Load training logs
log_file = Path(output_dir) / "trainer_state.json"

if log_file.exists():
    with open(log_file, 'r') as f:
        training_state = json.load(f)
    
    print("📊 Training Summary:\n")
    print(f"Total steps: {training_state.get('global_step', 'N/A')}")
    print(f"Best model found at step: {training_state.get('best_model_checkpoint', 'N/A')}")
    print(f"\nTraining completed successfully! ✅")
else:
    print("Training logs not found, but model was saved successfully!")

print(f"\n📂 Model location: {output_dir}")

## 🚀 Export for Ollama Deployment

In [ ]:
print("Merging LoRA weights with base model...\n")

# Merge LoRA weights
merged_model = model.merge_and_unload()
merged_output = "urdu-llama-3.2-merged"

merged_model.save_pretrained(merged_output)
tokenizer.save_pretrained(merged_output)

print(f"✓ Model merged and saved to {merged_output}")
print(f"\n📦 Next steps for Ollama deployment:")
print(f"  1. Download the '{merged_output}' folder")
print(f"  2. Convert to GGUF format (optional, for smaller file)")
print(f"  3. Create Modelfile in ollama/ directory")
print(f"  4. Run: ollama create urdu-llama -f ollama/Modelfile")
print(f"  5. Run: ollama run urdu-llama")

## ✨ Final Summary

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║                   🎉 TRAINING COMPLETE 🎉                          ║
╚════════════════════════════════════════════════════════════════════╝

✅ Successfully fine-tuned Llama 3.2 for Urdu!

📊 Training Summary:
  • Dataset: Urdu-English translation pairs
  • Model: meta-llama/Llama-3.2-1B-Instruct
  • Method: QLoRA (4-bit quantization)
  • Training examples: {0:,}
  • Validation examples: {1:,}
  • Total epochs: {2}

📁 Outputs:
  • LoRA adapter: urdu-llama-3.2-qLoRA-final/
  • Merged model: urdu-llama-3.2-merged/

🚀 Next Steps:
  1. Download the model checkpoint
  2. Deploy locally with Ollama
  3. Test on Urdu prompts
  4. Share results and iterate!

📖 Repository: https://github.com/danielosinor123-gif/llama-3.2-urdu

Happy fine-tuning! 🚀
""".format(len(train_dataset), len(val_dataset), CONFIG['num_epochs']))